# NYC Taxi Data Analysis

This notebook analyzes NYC Yellow Taxi trip data to examine how trip patterns between boroughs change over time.

We compare three different data access methods:
1. Loading individual monthly parquet files
2. Loading a single combined parquet file
3. Using DuckDB for SQL-based queries

## Cell 1: Imports and Setup

In [ ]:
import os
import time
from pathlib import Path

import duckdb
import pandas as pd
from dotenv import load_dotenv

from bettercode.taxi_utils import (
    combine_preprocessed_files,
    compute_borough_matrix,
    compute_borough_matrix_duckdb,
    create_duckdb_database,
    download_taxi_data,
    get_data_dirs,
    preprocess_all_files,
)

load_dotenv()
DATADIR = Path(os.getenv("DATADIR"))

## Cell 2: Download Data

In [ ]:
download_taxi_data(DATADIR, delay_between_downloads=30)

## Cell 3: Preprocess Data

In [ ]:
start_time = time.time()
preprocess_all_files(DATADIR)
end_time = time.time()
print(f"Preprocessing took {end_time - start_time:.2f} seconds")

## Cell 4: Combine into Single Parquet

In [ ]:
start_time = time.time()
combined_path = combine_preprocessed_files(DATADIR)
end_time = time.time()
print(f"Combining preprocessed files took {end_time - start_time:.2f} seconds")

## Cell 5: Create DuckDB Database

In [ ]:
db_path = DATADIR / "nyctaxi" / "taxi.duckdb"
start_time = time.time()
create_duckdb_database(combined_path, db_path)
end_time = time.time()
print(f"Creating DuckDB database took {end_time - start_time:.2f} seconds")

## Cell 6: Method 1 - Individual Files

Load each monthly parquet file and compute the borough matrix.

## Cell 6a: Load Time Comparison - Individual Files

Measure the time to load all individual parquet files without any computation.

In [ ]:
orig_dir, preproc_dir = get_data_dirs(DATADIR)
start_load = time.perf_counter()
individual_dfs = []
for file in sorted(preproc_dir.glob("*.parquet")):
    df = pd.read_parquet(file)
    individual_dfs.append(df)
time_load_individual = time.perf_counter() - start_load
print(f"Loading {len(individual_dfs)} individual files: {time_load_individual:.2f}s")

## Cell 6b: Load Time Comparison - Combined File

Measure the time to load the single combined parquet file.

In [ ]:
start_load = time.perf_counter()
combined_df = pd.read_parquet(combined_path)
time_load_combined = time.perf_counter() - start_load
print(f"Loading combined file: {time_load_combined:.2f}s")
print(f"\nSpeedup: {time_load_individual / time_load_combined:.2f}x faster")

## Cell 7: Method 1 - indivdiual files

In [ ]:
orig_dir, preproc_dir = get_data_dirs(DATADIR)
start = time.perf_counter()
results_method1 = {}
for file in sorted(preproc_dir.glob("*.parquet")):
    df = pd.read_parquet(file)
    results_method1[file.stem] = compute_borough_matrix(df)
time_method1 = time.perf_counter() - start
print(f"Method 1 (individual files): {time_method1:.2f}s")

## Cell 7: Method 2 - Single Parquet File

Load the combined parquet file and use groupby to compute matrices.

In [ ]:
start = time.perf_counter()
df = pd.read_parquet(combined_path)
results_method2 = {}
for (year, month), group in df.groupby(
    [df["tpep_pickup_datetime"].dt.year, df["tpep_pickup_datetime"].dt.month]
):
    results_method2[(year, month)] = compute_borough_matrix(group)
time_method2 = time.perf_counter() - start
print(f"Method 2 (combined parquet): {time_method2:.2f}s")

## Cell 8: Method 3 - DuckDB

Use DuckDB SQL queries to compute borough matrices.

In [ ]:
start = time.perf_counter()
con = duckdb.connect(str(db_path), read_only=True)
results_method3 = {}

# Get distinct year/month combinations
periods = con.execute("""
    SELECT DISTINCT YEAR(tpep_pickup_datetime) as year,
           MONTH(tpep_pickup_datetime) as month
    FROM taxi_trips ORDER BY year, month
""").fetchall()

for year, month in periods:
    results_method3[(year, month)] = compute_borough_matrix_duckdb(con, year, month)
con.close()
time_method3 = time.perf_counter() - start
print(f"Method 3 (DuckDB): {time_method3:.2f}s")

## Cell 9: Performance Comparison

In [ ]:
print("\n=== Performance Comparison ===")
print(f"Method 1 (individual files): {time_method1:.2f}s")
print(f"Method 2 (combined parquet): {time_method2:.2f}s")
print(f"Method 3 (DuckDB):           {time_method3:.2f}s")

## Cell 10: Sample Results

Display a sample borough matrix to verify the analysis.

In [ ]:
# Display a sample matrix from the most recent month
sample_key = max(results_method1.keys())
print(f"\nSample borough matrix for {sample_key}:")
results_method1[sample_key]